# 🤟 한국수어 인식 서비스
**데이터셋**: AI Hub 수어 영상 (국립국어원 표준 한국수어, 3,000 단어)  
**방법론**: 키포인트 JSON 파싱 → 정규화 → 머신러닝/딥러닝 분류  
**인식 단어**: 30개 (인사/감정/의사표현/일상명사/사람/동작)  
**출력**: 텍스트 + Gradio UI

In [ ]:
# ── 0. 환경 설정 ────────────────────────────────────────────
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

!pip install -q gradio
print('패키지 설치 완료')

In [ ]:
# ── 1. 라이브러리 및 전역 상수 ──────────────────────────────
import os, json, glob, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import joblib
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 인식 대상 30개 단어
TARGET_WORDS = [
    '안녕하세요','감사합니다','죄송합니다','괜찮아요','잠깐만요',
    '좋아요','싫어요','기뻐요','슬퍼요','화나요',
    '네','아니요','모르겠어요','도와주세요','아파요',
    '밥','물','화장실','집','병원',
    '나','너','엄마','아빠','친구',
    '먹다','마시다','가다','오다','자다',
]

DATA_ROOT  = '/content/aihub_data'
MODEL_PATH = '/content/sign_model.pkl'
print(f'디바이스: {DEVICE} | 인식 단어: {len(TARGET_WORDS)}개')

In [ ]:
# ── 2. 한글 폰트 설정 ───────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1
!fc-cache -fv > /dev/null 2>&1
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(5,1))
ax.text(0.5,0.5,'한글 폰트 테스트: 한국수어 인식', ha='center', va='center', fontsize=13)
ax.axis('off'); plt.show()
print('폰트 설정 완료')

In [ ]:
# ── 3. AI Hub aihubshell 설치 (선택사항 — Drive 방식 사용 시 불필요)
# Google Drive에서 파일을 가져오는 방식으로 진행합니다.

from google.colab import drive
drive.mount('/content/drive')

import os
# Drive에서 파일이 있는 폴더 확인
drive_root = '/content/drive/MyDrive'
print('Drive 연결 완료')
print('Drive 파일 목록:')
for f in os.listdir(drive_root):
    print(f'  {f}')

In [ ]:
# ── 5. Drive에서 파일 복사 및 압축 해제 ────────────────────
import zipfile, shutil, os
from pathlib import Path

# ↓ Drive에 올린 파일 경로 (폴더명이 다르면 수정)
MORPHEME_ZIP = '/content/drive/MyDrive/01_real_word_morpheme.zip'
KEYPOINT_ZIP = '/content/drive/MyDrive/01_real_word_keypoint.zip'

DATA_ROOT = '/content/aihub_data'
os.makedirs(DATA_ROOT, exist_ok=True)

# 압축 해제
for zip_path in [MORPHEME_ZIP, KEYPOINT_ZIP]:
    if os.path.exists(zip_path):
        print(f'압축 해제 중: {Path(zip_path).name} ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DATA_ROOT)
        print(f'  완료')
    else:
        print(f'파일 없음: {zip_path}')
        print('  → Drive 업로드 경로를 확인하세요')

print(f'\n압축 해제 완료')
!find /content/aihub_data -name "*.json" | wc -l
print('개의 JSON 파일 발견')

In [ ]:
# ── 5. JSON 키포인트 파일 다운로드 ──────────────────────────
# 위 셀 출력에서 확인한 keypoint JSON filekey로 교체
FILEKEY_KEYPOINT  = 'XXXXX'   # ← keypoint JSON filekey
FILEKEY_MORPHEME  = 'XXXXX'   # ← morpheme(단어 레이블) JSON filekey

!mkdir -p {DATA_ROOT}
!cd {DATA_ROOT} && ../aihubshell -mode d -datasetkey 7965 -filekey {FILEKEY_KEYPOINT},{FILEKEY_MORPHEME}
print('다운로드 완료')
!ls -lh {DATA_ROOT}

In [ ]:
# ── 6. 압축 해제 ─────────────────────────────────────────────
import zipfile, os

for zf in Path(DATA_ROOT).glob('*.zip'):
    print(f'압축 해제: {zf.name} ...')
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(DATA_ROOT)

json_files = list(Path(DATA_ROOT).rglob('*.json'))
print(f'\n총 JSON 파일: {len(json_files)}개')
for f in json_files[:5]:
    print(f'  {f}')

In [ ]:
# ── 7. JSON 구조 자동 탐지 ──────────────────────────────────
sample_path = json_files[0]
with open(sample_path, encoding='utf-8') as f:
    sample = json.load(f)

print('=== JSON 최상위 키 ===')
print(list(sample.keys()))
print()
print('=== 전체 구조 (500자) ===')
print(json.dumps(sample, ensure_ascii=False, indent=2)[:500])

In [ ]:
# ── 8. JSON 파싱 함수 (AI Hub 수어 영상 키포인트 형식) ────────
# AI Hub 키포인트 구조:
#   hand_right_keypoints_2d / hand_left_keypoints_2d : [[x,y,score] × 21]
#   pose_keypoints_2d                                 : [[x,y,score] × 25]
# 영상 1개 = 여러 프레임 → 프레임별 키포인트 리스트

def extract_word_from_json(data: dict) -> str | None:
    """JSON에서 수어 단어(레이블) 추출 — 키 이름 자동 탐지"""
    for key in ['word', 'label', 'sign', 'name', '단어', '수어단어']:
        if key in data:
            return str(data[key])
    # 중첩 구조 탐색
    for key in ['annotation', 'metadata', 'info', 'data']:
        if key in data and isinstance(data[key], dict):
            result = extract_word_from_json(data[key])
            if result:
                return result
    return None

def parse_keypoints_from_frame(frame: dict) -> np.ndarray | None:
    """프레임 1개에서 손 키포인트 42차원 벡터 추출"""
    # 오른손 우선, 없으면 왼손, 없으면 None
    right = frame.get('hand_right_keypoints_2d', [])
    left  = frame.get('hand_left_keypoints_2d',  [])

    pts = None
    if right and len(right) == 21:
        pts = np.array([[p[0], p[1]] for p in right], dtype=np.float32)
    elif left and len(left) == 21:
        pts = np.array([[p[0], p[1]] for p in left], dtype=np.float32)
        pts[:, 0] *= -1  # 왼손 x 반전 → 오른손 기준 통일
    else:
        return None

    # 정규화: 손목(0번) 기준 평행이동 + landmark 9 거리 스케일
    pts -= pts[0]
    scale = float(np.linalg.norm(pts[9]))
    if scale > 1e-6:
        pts /= scale
    return pts.flatten()

def json_to_feature(json_path: str, strategy: str = 'mean') -> tuple | None:
    """
    JSON 1개 → (단어, 특징벡터) 반환
    strategy: 'mean'  — 전체 프레임 평균 (정적 모델용, 42D)
              'seq'   — 프레임 시퀀스 반환 (LSTM용, T×42)
    """
    with open(json_path, encoding='utf-8') as f:
        data = json.load(f)

    word = extract_word_from_json(data)
    if not word:
        return None

    # 프레임 리스트 탐색
    frames = []
    for key in ['keypoints', 'frames', 'data', 'annotations']:
        if key in data and isinstance(data[key], list):
            frames = data[key]
            break
    if not frames:  # 단일 프레임 구조
        frames = [data]

    vecs = []
    for frame in frames:
        v = parse_keypoints_from_frame(frame)
        if v is not None:
            vecs.append(v)

    if not vecs:
        return None

    if strategy == 'mean':
        return word, np.mean(vecs, axis=0)  # 42D
    else:
        return word, np.array(vecs)          # T×42D

# 파싱 테스트
result = json_to_feature(str(json_files[0]))
if result:
    word, vec = result
    print(f'단어: {word}  |  벡터 형태: {vec.shape}')
else:
    print('파싱 실패 → 위 셀 7에서 JSON 구조 확인 후 수정 필요')

In [ ]:
# ── 9. 30개 단어 필터링 및 전체 데이터셋 구축 ──────────────
from tqdm.notebook import tqdm

rows, skipped_no_hand, skipped_word = [], 0, 0
TARGET_SET = set(TARGET_WORDS)

for jf in tqdm(json_files, desc='JSON 파싱'):
    result = json_to_feature(str(jf), strategy='mean')
    if result is None:
        skipped_no_hand += 1
        continue
    word, vec = result
    if word not in TARGET_SET:
        skipped_word += 1
        continue
    rows.append([word] + vec.tolist())

feat_cols = [f'{ax}{i}' for i in range(21) for ax in ('x','y')]
df = pd.DataFrame(rows, columns=['label'] + feat_cols)

print(f'\n추출 완료: {len(df):,}개')
print(f'손 미감지 제외: {skipped_no_hand}개 | 대상 외 단어 제외: {skipped_word}개')
print(f'\n단어별 샘플 수:')
vc = df['label'].value_counts()
print(vc.to_string())

missing = [w for w in TARGET_WORDS if w not in vc.index]
if missing:
    print(f'\n⚠ 데이터 없는 단어: {missing}')

df.to_csv('/content/landmarks_aihub.csv', index=False, encoding='utf-8-sig')
print('\n저장: /content/landmarks_aihub.csv')

In [ ]:
# ── 10. 데이터 분포 시각화 ───────────────────────────────────
vc = df['label'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(vc)))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 바 차트
axes[0].bar(vc.index, vc.values, color=colors, edgecolor='black', lw=0.6)
axes[0].set_title('단어별 샘플 수', fontsize=13, fontweight='bold')
axes[0].set_ylabel('샘플 수')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(vc.values):
    axes[0].text(i, v+1, str(v), ha='center', fontsize=8)

# 카테고리별 파이 차트
categories = {
    '인사/기본': ['안녕하세요','감사합니다','죄송합니다','괜찮아요','잠깐만요'],
    '감정':      ['좋아요','싫어요','기뻐요','슬퍼요','화나요'],
    '의사표현':  ['네','아니요','모르겠어요','도와주세요','아파요'],
    '일상명사':  ['밥','물','화장실','집','병원'],
    '사람':      ['나','너','엄마','아빠','친구'],
    '동작':      ['먹다','마시다','가다','오다','자다'],
}
cat_counts = {cat: sum(vc.get(w, 0) for w in words) for cat, words in categories.items()}
axes[1].pie(cat_counts.values(), labels=cat_counts.keys(),
            autopct='%1.1f%%', startangle=90,
            colors=plt.cm.Pastel1(np.linspace(0,1,6)))
axes[1].set_title('카테고리별 분포', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'총 샘플: {len(df):,}개 | 클래스: {df["label"].nunique()}개')

In [ ]:
# ── 11. 평균 손 모양 시각화 ──────────────────────────────────
CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17)
]

words_to_show = TARGET_WORDS[:10]
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('단어별 평균 손 모양 (AI Hub 데이터)', fontsize=14, fontweight='bold')

feat_cols = [f'{ax}{i}' for i in range(21) for ax in ('x','y')]
for ax, word in zip(axes.flatten(), words_to_show):
    subset = df[df['label']==word][feat_cols].values
    if len(subset) == 0:
        ax.set_title(f'{word}\n(데이터 없음)', fontsize=9); ax.axis('off'); continue
    pts = subset.mean(axis=0).reshape(21, 2)
    pts[:,1] *= -1
    for s,e in CONNECTIONS:
        ax.plot([pts[s,0],pts[e,0]], [pts[s,1],pts[e,1]], 'b-', lw=1.5, alpha=0.6)
    ax.scatter(pts[:,0], pts[:,1], c='red', s=25, zorder=5)
    ax.set_title(f'{word}\n(n={len(subset)})', fontsize=10, fontweight='bold')
    ax.set_aspect('equal'); ax.axis('off')

plt.tight_layout(); plt.show()

In [ ]:
# ── 12. 학습/검증/테스트 분할 ───────────────────────────────
feat_cols = [f'{ax}{i}' for i in range(21) for ax in ('x','y')]
X = df[feat_cols].values.astype(np.float32)
y = df['label'].values

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_tv, X_test, y_tv, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.125, stratify=y_tv, random_state=SEED)

print(f'학습:   {len(X_train):>6,}개  ({len(X_train)/len(X)*100:.1f}%)')
print(f'검증:   {len(X_val):>6,}개  ({len(X_val)/len(X)*100:.1f}%)')
print(f'테스트: {len(X_test):>6,}개  ({len(X_test)/len(X)*100:.1f}%)')
print(f'특징 차원: {X.shape[1]}D | 클래스: {len(le.classes_)}개')

In [ ]:
# ── 13. 머신러닝 모델 학습 (RF / SVM / MLP) ─────────────────
ml_models = {
    'RandomForest': Pipeline([
        ('sc', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=300, max_depth=None,
                                       random_state=SEED, n_jobs=-1))
    ]),
    'SVM (RBF)': Pipeline([
        ('sc', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=10, gamma='scale',
                   random_state=SEED, probability=True))
    ]),
    'MLP': Pipeline([
        ('sc', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(512, 256, 128),
                              activation='relu', max_iter=500,
                              random_state=SEED, early_stopping=True))
    ]),
}

ml_results = {}
for name, model in ml_models.items():
    print(f'학습: {name} ...')
    model.fit(X_train, y_train)
    val_acc  = accuracy_score(y_val,  model.predict(X_val))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    ml_results[name] = {'val': val_acc, 'test': test_acc, 'model': model}
    print(f'  검증: {val_acc:.4f}  |  테스트: {test_acc:.4f}')

best_ml = max(ml_results, key=lambda k: ml_results[k]['test'])
print(f'\n최고 ML 모델: {best_ml} ({ml_results[best_ml]["test"]:.4f})')

In [ ]:
# ── 14. LSTM 딥러닝 모델 (동적 수어 대응) ───────────────────
# 정적 벡터를 시퀀스처럼 reshape해서 LSTM 학습
# (실제 프레임 시퀀스 데이터가 있으면 ── 16. 시퀀스 데이터셋에서 교체)

class SignLSTM(nn.Module):
    def __init__(self, input_dim=42, hidden=128, layers=2, num_classes=30, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])

class SignDataset(Dataset):
    def __init__(self, X, y, seq_len=10):
        # 42D 벡터를 seq_len × (42//seq_len) 형태로 reshape
        self.X = torch.tensor(X.reshape(len(X), seq_len, -1), dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

SEQ_LEN = 6  # 42 / 7 = 6 features per step
# 42차원 → 6 timestep × 7 features
# reshape 가능한 조합: (6,7), (7,6), (14,3), (21,2) 등
SEQ_LEN, FEAT_DIM = 21, 2  # 21 관절 × 2(x,y) — 가장 직관적

train_ds = SignDataset(X_train, y_train, SEQ_LEN)
val_ds   = SignDataset(X_val,   y_val,   SEQ_LEN)
test_ds  = SignDataset(X_test,  y_test,  SEQ_LEN)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=128)
test_dl  = DataLoader(test_ds,  batch_size=128)

NUM_CLASSES = len(le.classes_)
model_lstm = SignLSTM(FEAT_DIM, 128, 2, NUM_CLASSES).to(DEVICE)
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.AdamW(model_lstm.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

print(f'LSTM 파라미터 수: {sum(p.numel() for p in model_lstm.parameters()):,}')

EPOCHS = 50
history = {'train_loss':[], 'val_acc':[]}
best_val, best_state = 0, None

for epoch in range(1, EPOCHS+1):
    model_lstm.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model_lstm(xb), yb)
        loss.backward(); optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    # 검증
    model_lstm.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in val_dl:
            preds += model_lstm(xb.to(DEVICE)).argmax(1).cpu().tolist()
            trues += yb.tolist()
    val_acc = accuracy_score(trues, preds)
    history['train_loss'].append(total_loss/len(train_dl))
    history['val_acc'].append(val_acc)

    if val_acc > best_val:
        best_val = val_acc
        best_state = {k:v.clone() for k,v in model_lstm.state_dict().items()}

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | loss: {total_loss/len(train_dl):.4f} | val_acc: {val_acc:.4f}')

model_lstm.load_state_dict(best_state)
preds_test = []
with torch.no_grad():
    for xb, yb in test_dl:
        preds_test += model_lstm(xb.to(DEVICE)).argmax(1).cpu().tolist()
lstm_test_acc = accuracy_score(y_test, preds_test)
print(f'\nLSTM 테스트 정확도: {lstm_test_acc:.4f}')

In [ ]:
# ── 19. 모델 저장 및 로컬 서비스용 다운로드 ─────────────────
import json as json_lib
from google.colab import files

joblib.dump(ml_results[best_ml]['model'], '/content/sign_model.pkl')
joblib.dump(le, '/content/label_encoder.pkl')
torch.save(model_lstm.state_dict(), '/content/sign_lstm.pth')

meta = {
    'best_ml_model':  best_ml,
    'ml_test_acc':    round(ml_results[best_ml]['test'], 4),
    'lstm_test_acc':  round(lstm_test_acc, 4),
    'num_classes':    NUM_CLASSES,
    'feature_dim':    42,
    'classes':        list(le.classes_),
    'target_words':   TARGET_WORDS,
    'dataset':        'AI Hub 수어 영상 (datasetSn=103)'
}
with open('/content/model_meta.json', 'w', encoding='utf-8') as f:
    json_lib.dump(meta, f, ensure_ascii=False, indent=2)

print('저장 완료')
print(f'최고 모델: {best_ml}  |  테스트 정확도: {ml_results[best_ml]["test"]*100:.1f}%')
print()
print('━━━ 로컬 서비스 연결 방법 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('아래 두 파일을 다운로드 후 sign_language_app/model/ 폴더에 넣으세요')
print('그 다음 python 04_app.py 실행')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

# 파일 다운로드
files.download('/content/sign_model.pkl')
files.download('/content/label_encoder.pkl')

In [ ]:
# ── 16. 전체 모델 성능 비교 ──────────────────────────────────
all_results = {**{k: v['test'] for k,v in ml_results.items()},
               'LSTM': lstm_test_acc}

names = list(all_results.keys())
accs  = [v*100 for v in all_results.values()]
colors_bar = ['#4472C4','#ED7D31','#A9D18E','#FF0000']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, accs, color=colors_bar[:len(names)], edgecolor='black', lw=0.7)
for bar, v in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_title('모델별 테스트 정확도 비교 (AI Hub 한국수어)', fontsize=13, fontweight='bold')
ax.set_ylabel('정확도 (%)', fontsize=11)
ax.set_ylim(0, 115)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

best_model_name = max(all_results, key=all_results.get)
print(f'최종 최고 모델: {best_model_name} ({all_results[best_model_name]*100:.1f}%)')

In [ ]:
# ── 17. 혼동 행렬 ────────────────────────────────────────────
# 최고 ML 모델 기준
best_ml_model = ml_results[best_ml]['model']
y_pred = best_ml_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
ko_labels = list(le.classes_)

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=ko_labels, yticklabels=ko_labels,
            linewidths=0.4, ax=ax)
ax.set_title(f'혼동 행렬 — {best_ml} (AI Hub 한국수어)', fontsize=14, fontweight='bold')
ax.set_xlabel('예측 레이블', fontsize=11)
ax.set_ylabel('실제 레이블', fontsize=11)
plt.xticks(rotation=40, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# ── 18. 분류 보고서 + 클래스별 정확도 ───────────────────────
print(f'=== {best_ml} 분류 보고서 ===')
print(classification_report(y_test, y_pred, target_names=ko_labels, digits=4))

per_class = cm.diagonal() / cm.sum(axis=1) * 100
sort_idx  = np.argsort(per_class)

fig, ax = plt.subplots(figsize=(13, 5))
colors_c = ['#37863D' if a>=90 else '#FFC000' if a>=70 else '#C00000'
            for a in per_class[sort_idx]]
sorted_labels = [ko_labels[i] for i in sort_idx]
bars = ax.barh(sorted_labels, per_class[sort_idx], color=colors_c, edgecolor='black', lw=0.5)
for bar, v in zip(bars, per_class[sort_idx]):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'{v:.1f}%', va='center', fontsize=9)
ax.axvline(90, color='green',  ls='--', alpha=0.6, label='90%')
ax.axvline(70, color='orange', ls='--', alpha=0.6, label='70%')
ax.set_title('단어별 인식 정확도 (낮은 순)', fontsize=13, fontweight='bold')
ax.set_xlabel('정확도 (%)')
ax.legend(); ax.set_xlim(0, 115)
plt.tight_layout(); plt.show()

In [ ]:
# ── 19. 모델 저장 ────────────────────────────────────────────
import json as json_lib

# 최고 ML 모델 저장
joblib.dump(ml_results[best_ml]['model'], '/content/sign_model.pkl')
joblib.dump(le, '/content/label_encoder.pkl')

# LSTM 저장
torch.save(model_lstm.state_dict(), '/content/sign_lstm.pth')

meta = {
    'best_ml_model':  best_ml,
    'ml_test_acc':    round(ml_results[best_ml]['test'], 4),
    'lstm_test_acc':  round(lstm_test_acc, 4),
    'num_classes':    NUM_CLASSES,
    'feature_dim':    42,
    'seq_len':        SEQ_LEN,
    'feat_dim_lstm':  FEAT_DIM,
    'classes':        list(le.classes_),
    'target_words':   TARGET_WORDS,
    'dataset':        'AI Hub 수어 영상 (datasetkey=7965)'
}
with open('/content/model_meta.json', 'w', encoding='utf-8') as f:
    json_lib.dump(meta, f, ensure_ascii=False, indent=2)

print('저장 완료:')
print('  /content/sign_model.pkl    (ML 모델)')
print('  /content/label_encoder.pkl')
print('  /content/sign_lstm.pth     (LSTM 모델)')
print('  /content/model_meta.json')

In [ ]:
# ── 20. Gradio 추론 인터페이스 ───────────────────────────────
import gradio as gr
import mediapipe as mp
import cv2
import numpy as np
from PIL import Image

!pip install -q "mediapipe==0.10.14"

# 모델 로드
infer_model = joblib.load('/content/sign_model.pkl')
infer_le    = joblib.load('/content/label_encoder.pkl')
with open('/content/model_meta.json', encoding='utf-8') as f:
    infer_meta = json_lib.load(f)

mp_hands = mp.solutions.hands

def normalize_pts(pts):
    pts = pts.copy()
    pts -= pts[0]
    scale = float(np.linalg.norm(pts[9]))
    if scale > 1e-6: pts /= scale
    return pts

def predict(pil_img):
    img = np.array(pil_img.convert('RGB'))
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    with mp_hands.Hands(static_image_mode=True, max_num_hands=1,
                        min_detection_confidence=0.5) as hands:
        result = hands.process(img_rgb)

    if not result.multi_hand_landmarks:
        return '❌ 손이 감지되지 않았습니다\n손을 카메라 정면에 보여주세요'

    hl  = result.multi_hand_landmarks[0]
    pts = np.array([[lm.x, lm.y] for lm in hl.landmark], dtype=np.float32)

    # 왼손이면 x 반전
    if result.multi_handedness[0].classification[0].label == 'Left':
        pts[:, 0] *= -1
    pts = normalize_pts(pts)
    vec = pts.flatten().reshape(1, -1)

    pred_enc  = infer_model.predict(vec)[0]
    pred_prob = infer_model.predict_proba(vec)[0]
    pred_word = infer_le.inverse_transform([pred_enc])[0]
    conf      = pred_prob.max() * 100

    top3 = pred_prob.argsort()[-3:][::-1]
    top3_str = '\n'.join(
        f'  {i+1}. {infer_le.classes_[idx]}  ({pred_prob[idx]*100:.1f}%)'
        for i, idx in enumerate(top3)
    )

    return f'🤟 인식 결과: {pred_word}\n신뢰도: {conf:.1f}%\n\nTop-3 예측:\n{top3_str}'

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type='pil', label='수어 이미지 업로드'),
    outputs=gr.Textbox(label='인식 결과', lines=8),
    title='🤟 한국수어 인식 서비스',
    description=(
        f'AI Hub 표준 한국수어 데이터 기반 · 인식 단어 {len(TARGET_WORDS)}개\n'
        f'모델: {infer_meta["best_ml_model"]} · '
        f'정확도: {infer_meta["ml_test_acc"]*100:.1f}%'
    ),
    flagging_mode='never'
)
demo.launch(share=True)

## 📋 진행 현황

| 단계 | 내용 | 상태 |
|------|------|------|
| ✅ 0~3 | 환경 설정, 패키지, 폰트 | 완료 |
| ⏳ 4~6 | AI Hub 로그인, 파일 목록, 다운로드 | **직접 실행 필요** |
| ⏳ 7~9 | JSON 파싱, 30개 단어 필터링 | 다운로드 후 자동 실행 |
| ⏳ 10~18 | 시각화, 모델 학습, 평가 | 자동 실행 |
| ⏳ 19~20 | 모델 저장, Gradio UI | 자동 실행 |

### 직접 해야 할 것 (셀 4~6)
1. 셀 4: `AIHUB_ID`, `AIHUB_PW` 입력 후 실행 → 파일 목록 확인
2. 셀 5: 목록에서 JSON filekey 번호 → `FILEKEY_KEYPOINT`, `FILEKEY_MORPHEME` 교체
3. 이후 전체 실행 (`런타임 → 모두 실행`)